# 2015 Flight Delays and Cancellations

## Imports

Preferably, place any additional imports you will need throughout the notebook in the cell below.

In [1]:
import pandas as pd
from numpy import int64, float64

## Extract and Load Data

In [ ]:
AIRLINES_CSV_PATH = 'airlines.csv'
AIRPORTS_CSV_PATH = 'airports.csv'
FLIGHTS_CSV_PATH = 'flights.csv'

### TODO 1

Iterate over the csv_paths, table_names, and index_cols simultaneously using zip. This will give you access to each CSV path, its corresponding table name, and the index column in each iteration. 

For each iteration, implement the code where the data is loaded into a pandas DataFrame from the provided csv path (`pd.read_csv` function) using the respective value of index_col for the `index_col` parameter of the `pd.read_csv`. Store the dataframe as a value in the `df_dict` under the `table_name` as key.

In [3]:
def generate_pandas_df_dict(csv_paths, table_names=None, index_cols=None):
    if table_names is None:
        table_names = csv_paths
    if index_cols is None:
        index_cols = (None,) * len(csv_paths)
    df_dict = {}
    for csv_path, table_name, index_col in zip(csv_paths, table_names, index_cols):
        df_dict[table_name] = pd.read_csv(csv_path, index_col=index_col, low_memory=False)
    return df_dict

## Show Tables

### TODO 2

Implement the code so that the function prints output of the following shape:

    === TABLES ===
    - airlines (14 rows, 1 columns)
    - airports (322 rows, 6 columns)
    - flights (5819079 rows, 31 columns)
    TOTAL 3

Hint: Use the 'shape' attribute of each Dataframe to get the rows and columns.    

In [4]:
def show_tables(df_dict):
    print('=== TABLES ===')
    for name, df in sorted(df_dict.items()):
        print(f"- {name} ({df.shape[0]} rows, {df.shape[1]} columns)")
    print(f'TOTAL {len(df_dict)}')

## Describe Table

### TODO 3
Complete the code in the `if` branch with max, min and mean functions of the Dataframe's column i.e., replace
<MAX_FUNCTION>,<MIN_FUNCTION> and <MEAN_FUNCTION> with appropriate functions. 

Finish the `else` branch so that a column that is not int64 or float64 will have the following shape:

    - TAIL_NUMBER object (5804358/5819079) [4898 unique]

Note that for the count of unique objects an absenting value counts as one. For example, if there is a column where lines are either assigned with value "red" or value "blue", and there is a couple of line where the value is not assigned, then the column should be reported as having three unique values.

In [5]:
def describe_table(df):
    print('=== COLUMNS ===')
    for column, dtype in sorted(zip(df.columns, df.dtypes)):
        series = df[column]
        if dtype == int64 or dtype == float64:
            print(f'- {column} {dtype} ({series.count()}/{series.size}) ' +
                  f'[Max: {series.max()}, Mean: {series.mean()}, Min: {series.min()}]')
        else:
            print(f'- {column} {dtype} ({series.count()}/{series.size}) ' +
                  f'[{len(series.unique())} unique]')
    print(f'TOTAL {df.shape[0]}')

## Identify Useless Columns

### TODO 4

Add one additional check that will assess if the series has only a single value. If yes the following should be printed (COLUMN_NAME will replaced by the actual name of the column):

    The column COLUMN_NAME appears to have only one value.

Hint: Think something on the lines of a method that gives unique values having a parameter that cleans NaN data.

In [ ]:
def assess_useless_columns(df):
    for column in df.columns:
        series = df[column]
        if series.count() == 0:
            print(f'The column {column} appears to have no values.')
        if series.nunique(dropna=False) == 1:
            print(f'The column {column} appears to have only one value.')

## Delays Per Airline

The function `generate_delays_per_airline_csv` produces the following csv stored at the `csv_path`:

    AIRLINE,ARRIVAL_DELAY
    Spirit Air Lines,14.471799501705833
    Frontier Airlines Inc.,12.504706404706404
    JetBlue Airways,6.677860800940307
    Atlantic Southeast Airlines,6.585378691739733
    American Eagle Airlines Inc.,6.457873460764516
    Skywest Airlines Inc.,5.845652151300072
    United Air Lines Inc.,5.431593935741549
    Virgin America,4.737705721003135
    Southwest Airlines Co.,4.3749636792570525
    US Airways Inc.,3.7062088424131026
    American Airlines Inc.,3.4513721447256764
    Hawaiian Airlines Inc.,2.023092805197196
    Delta Air Lines Inc.,0.18675361236390797
    Alaska Airlines Inc.,-0.9765630924118783

Uncomment the commented lines below and fill the values for <COLUMN_1> and <COLUMN_2>. Look at the output above, observe the column names and use them to complete the code. 

In [ ]:
def generate_delays_per_airline_csv(df_dict, csv_path):
    delays_per_airline = pd.merge(
        left=df_dict['flights'][['AIRLINE','ARRIVAL_DELAY']].groupby(['AIRLINE']).mean(),
        right=df_dict['airlines'],
        left_index=True,
        right_index=True
    )[['AIRLINE', 'ARRIVAL_DELAY']]
    delays_per_airline.sort_values(by='ARRIVAL_DELAY', ascending=False, ignore_index=True, inplace=True)
    delays_per_airline.to_csv(csv_path, index=False)

## Delays Per Arrival Airport

### TODO 5

Implement the `generate_delays_per_arriving_airport_csv` function that works similarly to the `generate_delays_per_airline_csv`. However, instead of listing the average dalays per airline list them per airport where the flight arrives. The output should contain the name of the arriving airport in the first column and the average delay in the second.

In [ ]:
def generate_delays_per_arriving_airport_csv(df_dict, csv_path):
    delays_per_airline = pd.merge(
        left=df_dict['flights'][['AIRLINE','ARRIVAL_DELAY']].groupby(['AIRLINE']).mean(),
        right=df_dict['airlines'],
        left_on='DESTINATION_AIRPORT',
        right_on='IATA_CODE'
    )[['AIRLINE', 'ARRIVAL_DELAY']]
    delays_per_airline.sort_values(by='ARRIVAL_DELAY', ascending=False, ignore_index=True, inplace=True)
    delays_per_airline.to_csv(csv_path, index=False) 